# z/OS MIPS Prediction - Model Training

This notebook demonstrates the complete training pipeline for MIPS prediction models.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import MIPSDataLoader
from src.preprocessing import MIPSPreprocessor
from src.features import MIPSFeatureEngineering
from src.models.regression import RegressionModelFactory, BaselinePredictor
from src.models.classification import ClassificationModelFactory
from src.evaluation import ModelEvaluator
from src.visualization import MIPSVisualizer
from src.training import MIPSTrainingPipeline

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load and Prepare Data

In [ ]:
# Load data
data_path = '../data/sample/sample_mips_data.csv'
loader = MIPSDataLoader(data_path)
data = loader.load_csv()

print(f"Loaded {len(data)} records")
data.head()

In [ ]:
# Split data
train_data, test_data = loader.split_train_test(test_size=0.2, random_state=42)

print(f"Training set: {len(train_data)} samples")
print(f"Test set: {len(test_data)} samples")

## 2. Feature Engineering

In [ ]:
# Apply feature engineering
feature_eng = MIPSFeatureEngineering()

train_data_fe = feature_eng.create_all_features(train_data.copy())
test_data_fe = feature_eng.create_all_features(test_data.copy())

print(f"Original features: {len(train_data.columns)}")
print(f"After feature engineering: {len(train_data_fe.columns)}")
print(f"\nNew features created: {len(feature_eng.created_features)}")

## 3. Preprocessing

In [ ]:
# Split features and target
X_train, y_train = loader.get_feature_target_split(train_data_fe)
X_test, y_test = loader.get_feature_target_split(test_data_fe)

# Preprocess
preprocessor = MIPSPreprocessor(scaler_type='standard')
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

print(f"Training features shape: {X_train_scaled.shape}")
print(f"Test features shape: {X_test_scaled.shape}")

## 4. Train Regression Models

In [ ]:
# Create baseline models
models = RegressionModelFactory.create_baseline_models()

# Add baselines
models['baseline_mean'] = BaselinePredictor('mean')
models['baseline_median'] = BaselinePredictor('median')

print(f"Created {len(models)} models")

In [ ]:
# Train and evaluate all models
results = {}
evaluator = ModelEvaluator()

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # Evaluate
    train_metrics = evaluator.evaluate_regression(y_train, y_train_pred)
    test_metrics = evaluator.evaluate_regression(y_test, y_test_pred)
    
    results[name] = {
        'model': model,
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'y_pred': y_test_pred
    }
    
    print(f"  Train RMSE: {train_metrics['rmse']:.2f}")
    print(f"  Test RMSE: {test_metrics['rmse']:.2f}")
    print(f"  Test R²: {test_metrics['r2']:.4f}")

## 5. Compare Model Performance

In [ ]:
# Create comparison DataFrame
comparison = evaluator.compare_models(results, metric='rmse', mode='regression')
print("\nModel Comparison (RMSE):")
print(comparison)

In [ ]:
# Visualize comparison
viz = MIPSVisualizer(output_dir='../results')
viz.plot_model_comparison(comparison, metric='rmse', title='Model Comparison - RMSE')

In [ ]:
# R² comparison
r2_comparison = evaluator.compare_models(results, metric='r2', mode='regression')
r2_comparison = r2_comparison.sort_values('test_r2', ascending=False)
print("\nModel Comparison (R²):")
print(r2_comparison)

## 6. Analyze Best Model

In [ ]:
# Get best model
best_model_name = comparison.iloc[0]['model']
best_result = results[best_model_name]

print(f"\nBest Model: {best_model_name}")
print("\nTest Metrics:")
evaluator.print_regression_summary(best_result['test_metrics'], best_model_name)

In [ ]:
# Predictions vs Actual
viz.plot_predictions_vs_actual(
    y_test, 
    best_result['y_pred'],
    title=f"{best_model_name} - Predictions vs Actual"
)

In [ ]:
# Residual analysis
viz.plot_residuals(
    y_test,
    best_result['y_pred'],
    title=f"{best_model_name} - Residual Analysis"
)

In [ ]:
# Feature importance (if available)
if hasattr(best_result['model'], 'get_feature_importance'):
    importance = best_result['model'].get_feature_importance(X_train.columns.tolist())
    if not importance.empty:
        print("\nTop 20 Most Important Features:")
        print(importance.head(20))
        
        viz.plot_feature_importance(
            importance,
            top_n=20,
            title=f"{best_model_name} - Feature Importance"
        )

## 7. Using the Training Pipeline

In [ ]:
# Alternative: Use the complete training pipeline
config = {
    'data_path': '../data/sample/sample_mips_data.csv',
    'test_size': 0.2,
    'random_state': 42,
    'feature_engineering': True,
    'save_models': True,
    'output_dir': '../models',
    'results_dir': '../results'
}

pipeline = MIPSTrainingPipeline(config)
pipeline_results = pipeline.run_full_pipeline(mode='regression')

print("\nPipeline completed!")
print(f"Best model: {pipeline.best_model_name}")

## Summary

This notebook demonstrated:
1. Loading and preparing z/OS MIPS data
2. Applying feature engineering
3. Training multiple linear regression models
4. Comparing model performance
5. Analyzing the best model
6. Using the complete training pipeline

The best models beat the baseline predictors and provide accurate MIPS consumption predictions.